In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini 3.8 Flash Text-to-Speech Generation

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Faudio%2Fspeech%2Fgetting-started%2Fgemini_3_8_flash_tts.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/gemini_3_8_flash_tts.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author |
| --- |
| [Katie Nguyen](https://github.com/katiemn) |

## Overview

### Gemini 3.8 Flash TTS and Gemini 3.8 Flash-Lite TTS

Gemini 3.8 Flash TTS and Gemini 3.8 Flash-Lite TTS on Agent Platform give application developers access to Google's cutting-edge speech generation. Gemini 3.8 Flash TTS provides studio-grade audio with nuanced, expressive control, and Gemini 3.8 Flash-Lite TTS provides fast, cost-efficient speech generation for production workloads.

In this tutorial, you will learn how to use the Google Gen AI SDK for Python to:
- Select a voice:
    - Prebuilt voices
    - Extended Voice Library
    - Prompted voices
    - Replicated voices
- Manage stored voices in your project
- Convert text-to-speech:
  - Using prebuilt and prompted voices
  - Control delivery with styles and audio tags
  - Generate speech in different languages
  - Define multiple speakers

## Get started

### Install Google Gen AI SDK for Python

In [ ]:
%pip install --upgrade --quiet google-genai ipywebrtc

### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the following cell to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [ ]:
import io
import os
import subprocess
import wave
import base64
import google.auth
import numpy as np
from IPython.display import Audio, display
from google import genai
from google.genai import types
from ipywebrtc import AudioRecorder, CameraStream

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
)

### Define a helper function to play audio

In [ ]:
def play_audio(response):
    audio_bytes = response.candidates[0].content.parts[0].inline_data.data
    display(Audio(data=audio_bytes))

### Load the models

This notebook uses two TTS models:

- **Gemini 3.8 Flash TTS** (`gemini-3.8-flash-tts`): Use when acoustic fidelity, nuanced acting, and expressive control are the top priority. It suits studio-grade creative work, complex multi-speaker dialogue, frequent vocal-burst tags, difficult pronunciations, regional or minority dialects, and long-form narration that needs a stable voice and room tone.
- **Gemini 3.8 Flash-Lite TTS** (`gemini-3.8-flash-lite-tts`): Use for fast, cost-efficient production workloads. It suits high-volume batch production, conversational voice agent cascades, read-aloud features, voice replication, and everyday single-speaker speech across major languages.

In [ ]:
speech_model = "gemini-3.8-flash-tts"
lite_speech_model = "gemini-3.8-flash-lite-tts"

## Select a voice

Gemini TTS on Agent Platform supports four types of voices:

- **Prebuilt voices**: 30 ready-to-use voices, selected by name.
- **Extended Voice Library**: more than 2,000 curated preset voices, selected by ID.
- **Prompted voices**: voices you design from a text description, stored in your project.
- **Voice replication**: voices replicated from a recording of a speaker, either stored in your project or returned as a voice key.

In this section, you'll explore these options and create a voice. In the next section, you'll use these voices to generate audio.

### Prebuilt voices

Gemini TTS includes 30 prebuilt voices, such as `Fenrir`, `Despina`, `Charon`, and `Leda`. See the full list of [voice options](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/text-to-speech/overview#prebuilt-voices).

### Extended Voice Library

Beyond the 30 prebuilt voices, the Extended Voice Library provides more than 2,000 curated preset voices across languages, regional accents, character personas, and use cases such as audiobooks, conversational agents, and news.

To browse the library, call the Voices API `list` method. Each library voice includes its `id` and metadata such as `language_code`, `accent`, `gender`, `pitch`, `persona`, and `description`. To narrow the results, use the following filters:

- `type_`: The voice source: `prebuilt` for prebuilt and library voices, `prompted` for designed voices, or `replicated` for replicated voices. If you pass more than one value, the method returns voices that match any of them.
- `search`: Text to match, case-insensitively, against each voice's display name and description.

The following example searches the library for podcast voices.

In [ ]:
response = client.voices.list(type_=["prebuilt"], search="podcaster", page_size=50)
for voice in response.voices or []:
    print(voice.id, voice.language_code, voice.accent, voice.gender, voice.description)

Save the ID of a library voice to use when generating audio.

In [ ]:
library_voice_id = response.voices[0].id
print(f"Library voice ID: {library_voice_id}")

### Prompted voices

With prompted voices, you describe the voice you want in natural language, and the model creates it. Set `type` to `VOICE_TYPE_PROMPTED` and provide the description in `prompted.input`, along with the voice's `display_name`, `gender`, and `language_code`. Setting `store=True` saves the voice to your project so that you can reuse it by ID.

In [ ]:
created_voice = client.voices.create(
    store=True,
    voice={
        "type": "VOICE_TYPE_PROMPTED",
        "display_name": "Cheerful Weather Reporter",
        "gender": "female",
        "language_code": "en-US",
        "prompted": {
            "input": (
                """An enthusiastic local TV weather reporter in her 30s. She speaks with bright, bubbly energy."""
            )
        },
    },
)

print(f"Created voice ID: {created_voice.id}")

Use the `get` method to retrieve a voice by ID. For a designed voice, the response also includes the voice sample in the `sample_audio` field.

In [ ]:
voice = client.voices.get(id=created_voice.id)
print(voice.id, voice.display_name, voice.type)

audio_bytes = base64.b64decode(voice.sample_audio.data)
audio_array = np.frombuffer(audio_bytes, dtype="<i2")
display(Audio(data=audio_array, rate=24000))

### Voice replication

With voice replication, you create a voice from a short recording of a real speaker. Each request to create a replicated voice needs two recordings of the same adult speaker:

- **Reference audio** (`source_audio`): 10 to 30 seconds of clean, natural speech from the speaker whose voice you want to replicate. Longer samples generally produce better results.
- **Consent audio** (`consent_audio`): A recording of the same speaker reading the consent statement for their language word for word.

When you create a replicated voice, the store field selects one of two storage modes:

- **Stored voice** (`store=True`): Google stores the replicated voice in your project and returns a voice ID. You can list, get, and delete stored voices with the Voices API. These voices expire 1 year after last use.
- **Voice replication key** (`store=False`): Google doesn't store the replicated voice. The service returns an encrypted voice replication key, and your application stores the key. Use this mode if your workload requires that Google doesn't keep voice profiles. The key will expire 7 days after creation.

#### Record audio

In the following cells, you'll record the reference and consent audio with your microphone using the `AudioRecorder` widget. The following helper converts each recording to a 24 kHz mono WAV file.

If you already have recordings, set `SOURCE_AUDIO_PATH` and `CONSENT_AUDIO_PATH` to your audio files instead, and the helper loads them from disk.

In [ ]:
SOURCE_AUDIO_PATH = ""  # @param {type: "string", placeholder: "[optional-path-to-source-audio]"}
CONSENT_AUDIO_PATH = ""  # @param {type: "string", placeholder: "[optional-path-to-consent-audio]"}

if "google.colab" in sys.modules:
    from google.colab import output

    output.enable_custom_widget_manager()


def convert_to_wav(audio_bytes, max_seconds=None):
    duration = ["-t", str(max_seconds)] if max_seconds else []
    pcm = subprocess.run(
        ["ffmpeg", "-i", "pipe:0", *duration, "-ar", "24000", "-ac", "1", "-f", "s16le", "pipe:1"],
        input=audio_bytes,
        capture_output=True,
        check=True,
    ).stdout
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(24000)
        wav_file.writeframes(pcm)
    return buffer.getvalue()


def load_audio(path, recorder, max_seconds=None):
    if path:
        with open(path, "rb") as f:
            audio_bytes = f.read()
    else:
        audio_bytes = recorder.audio.value
        if not audio_bytes:
            raise ValueError("No recording found. Record audio with the widget first.")
    return convert_to_wav(audio_bytes, max_seconds)

#### Record the reference audio

Run the following cell to display the recorder. Click the record button, read the passage below in a natural speaking voice for 10 to 30 seconds, then click the button again to stop. You might need to run the cell twice after granting microphone permission.

> The morning fog had just begun to lift over the harbor when the first fishing boats returned. Gulls circled overhead, calling to one another, while the smell of salt and fresh bread drifted from the bakery on the corner. It was the kind of quiet, ordinary day that people rarely remember, and yet, years later, it was the one I thought about most.

In [ ]:
source_recorder = AudioRecorder(
    stream=CameraStream(constraints={"audio": True, "video": False})
)
source_recorder

Once you've stopped recording, convert the reference audio to WAV. Anything past 30 seconds is trimmed.

In [ ]:
source_wav = load_audio(SOURCE_AUDIO_PATH, source_recorder, max_seconds=30)

#### Record the consent audio

Run the following cell to display the recorder. Click the record button, read the consent statement word for word, then click the button again to stop:

> "I am the owner of this voice and have consented to the creation of a synthetic model of my voice through the use of Google Cloud."

Check out the docs to view the full list of supported languages and [consent statements](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/text-to-speech/voice-replication#consent-statements).

In [ ]:
consent_recorder = AudioRecorder(
    stream=CameraStream(constraints={"audio": True, "video": False})
)
consent_recorder

Once you've stopped recording, convert the consent audio to WAV.

In [ ]:
consent_wav = load_audio(CONSENT_AUDIO_PATH, consent_recorder)

#### **Option 1:** Create a stored replicated voice

Set `type` to `VOICE_TYPE_REPLICATED` and pass both recordings, base64-encoded, in the `replicated` field. With `store=True`, the voice is saved to your project and you can reuse it by its ID.

In [ ]:
source_b64 = base64.b64encode(source_wav).decode("utf-8")
consent_b64 = base64.b64encode(consent_wav).decode("utf-8")

replicated_voice = client.voices.create(
    store=True,
    voice={
        "type": "VOICE_TYPE_REPLICATED",
        "display_name": "Replicated speaker",
        "replicated": {
            "source_audio": {"mime_type": "audio/wav", "data": source_b64},
            "consent_audio": {"mime_type": "audio/wav", "data": consent_b64},
        },
    },
)

print(f"Created voice ID: {replicated_voice.id}")

#### **Option 2:** Generate a voice key

If you don't want to store the voice in your project, set `store=False`. Instead of a voice ID, the response returns a voice key. Store the key in your application and pass it in the `voice` field when you generate audio.

In [ ]:
replicated_voice_key = client.voices.create(
    store=False,
    voice={
        "model": speech_model,
        "type": "VOICE_TYPE_REPLICATED",
        "replicated": {
            "source_audio": {"mime_type": "audio/wav", "data": source_b64},
            "consent_audio": {"mime_type": "audio/wav", "data": consent_b64},
        },
    },
)

voice_key = replicated_voice_key.key
print(f"Voice key: {voice_key}")

### Manage stored voices

You can list, get, and delete the voices stored in your project, including designed voices and stored replicated voices.

The `list` method returns the voices stored in your project, followed by the prebuilt and Extended Voice Library voices. To list only the voices stored in your project, set the `type_` filter to `prompted` and `replicated`.

In [ ]:
for voice in client.voices.list(type_=["prompted", "replicated"]).voices or []:
    print(voice.id, voice.display_name, voice.type)

To delete a stored voice, use the `delete` method. You'll delete the voices you created in the clean up section at the end of this notebook, after using it to generate audio.

## Generate audio

Now that you've selected your voices, you'll use them to generate audio. The following examples show how to use each voice option with the model that best fits the task.

By default, all audio generated with Gemini 3.8 Flash TTS utilizes [SynthID](https://deepmind.google/technologies/synthid/).

### Convert text to speech

To use a prebuilt voice, pass its name in the `VoiceConfig`. This example uses Gemini 3.8 Flash-Lite TTS, which is well suited for everyday single-speaker speech.

In [ ]:
prompt = """
Hey there! How are you doing today?
"""

response = client.models.generate_content(
    model=lite_speech_model,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(voice="Fenrir")
        ),
    ),
)

play_audio(response)

### Control delivery with a style

Put emotion, delivery style, pacing, and volume that apply to a whole turn in `speech_metadata.style`. For example, "whispered urgently", "out of breath", or "warm and enthusiastic". You can also name the speaker with `speech_metadata.speaker`.

To use a voice from the Extended Voice Library, pass its ID in the `voice` field of the `VoiceConfig`.

In [ ]:
response = client.models.generate_content(
    model=lite_speech_model,
    contents=types.Content(
        role="user",
        parts=[
            types.Part(
                text=(
                    """Welcome back to the show! Today we're talking about the best hiking trails for beginners, so grab your boots and let's get going."""
                ),
                speech_metadata=types.SpeechMetadata(
                    speaker="Podcaster", style="cheerful and friendly"
                ),
            )
        ],
    ),
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(voice=library_voice_id)
        ),
    ),
)

play_audio(response)

### Add audio tags for point-in-time events

While a style applies to a whole turn, audio tags control point-in-time events. An audio tag is a non-speech vocal burst, breath, or pause in an angle bracket inserted directly into your prompt, such as `<gasp>` or `<cheer>`.

**Note:** For highest audio quality, stick to human vocalizations rather than non-vocal sound effects.

You can experiment with different tags. Here is a list of examples to get started:

| Audio Tag | Audio Tag | Audio Tag | Audio Tag | Audio Tag |
| --- | --- | --- | --- | --- |
| `<argh>` | `<breath>` | `<exhales>` | `<cackle>` | `<cheer>` |
| `<chuckle>` | `<cough>` | `<cry>` | `<gasp>` | `<giggle>` |
| `<groan>` | `<growl>` | `<grunt>` | `<grr>` | `<hiss>` |
| `<laugh>` | `<moan>` | `<pant>` | `<pff>` | `<phew>` |
| `<scream>` | `<shout>` | `<shriek>` | `<sigh>` | `<sneeze>` |
| `<snicker>` | `<snort>` | `<sob>` | `<throat-clearing>` | `<tsk>` |
| `<whimper>` | `<whispers>` | `<yawn>` | `<short pause>` | `<long pause>` |

In [ ]:
prompt = """
<sigh> I have absolutely no idea why this variable is returning as undefined, <groan> and I've been staring at this same block of code until my eyes are literally crossing! <shout> Just work already! <whispers> Wait... is it really just a missing semicolon right there on line forty-two? <laugh> I can't believe I spent two hours debugging a simple typo!
"""

response = client.models.generate_content(
    model=speech_model,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(voice=created_voice.id)
        ),
    ),
)

play_audio(response)

### Generate speech in different languages

The TTS models detect the input language automatically. Check the documentation to learn more about the list of [supported languages](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/text-to-speech/overview#languages).

Gemini 3.8 Flash-Lite TTS works well for everyday speech across major languages. For regional or minority dialects, use Gemini 3.8 Flash TTS.

In [ ]:
prompt = """
¡Hola! ¿Quieres hacer algo hoy?
"""

response = client.models.generate_content(
    model=lite_speech_model,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(voice="Despina")
        ),
    ),
)

play_audio(response)

### Generate speech with a replicated voice

To use a replicated voice, pass its voice key or its stored voice ID in the `voice` field of the `VoiceConfig`.

In [ ]:
prompt = """
This is my replicated voice, generated with Gemini 3.8 Flash TTS.
"""

response = client.models.generate_content(
    model=speech_model,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(voice=replicated_voice.id)
        ),
    ),
)

play_audio(response)

### Define multiple speakers

For multi-speaker audio clips, include a `MultiSpeakerVoiceConfig` in your request. You can include up to 2 different speakers, each with their own `SpeakerVoiceConfig` that includes the `speaker` name and a `VoiceConfig`.

Each speaker can use any voice option.

In [ ]:
response = client.models.generate_content(
    model=speech_model,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part(
                    text="How are you doing today Katie?",
                    speech_metadata=types.SpeechMetadata(
                        speaker="Ryan",
                        style="cheerful and friendly",
                    ),
                ),
                types.Part(
                    text="Not too bad, how about you?",
                    speech_metadata=types.SpeechMetadata(
                        speaker="Katie",
                        style="calm and relaxed",
                    ),
                ),
            ],
        )
    ],
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                speaker_voice_configs=[
                    types.SpeakerVoiceConfig(
                        speaker="Ryan",
                        voice_config=types.VoiceConfig(voice="Sadaltager"),
                    ),
                    types.SpeakerVoiceConfig(
                        speaker="Katie",
                        voice_config=types.VoiceConfig(voice="Leda"),
                    ),
                ]
            )
        ),
    ),
)
play_audio(response)

## Clean up

Delete the designed and replicated voices you stored in your project. Voice keys aren't stored in your project, so there's nothing to delete for them.

In [ ]:
client.voices.delete(id=created_voice.id)
client.voices.delete(id=replicated_voice.id)